In [1]:
import xarray as xr
import pandas as pd
from pathlib import Path
from dask import delayed, compute
import multiprocessing
import os
import warnings
import rechunker

# Use this notebook to create Zarr stores for each of the datasets
Should you wish not to use the default locations set by the tutorial notebook, you will have to provide the path to the netcdf files and the path to where you want the zarr store to exist.<br>

It is worth noting that each netcdf file contains all data for a single station, and there are two functions defined below that are necessary for converting our station data to zarr:
- `preprocess_station`
- `process_all_to_zarr`



There is a final function `load_combined_dataset` that is used to verify the conversion succeeded and that all station data can be loaded together.


# Preprocess NetCDF Files
Purpose:
- Opens a single NetCDF file, standardizes it, and prepares it for analysis.

Key steps:
- Loads the NetCDF file as an xarray Dataset.
- Drops the input_station_id variable if present (to avoid object dtype issues).
- Assigns a station_id coordinate from the file’s attributes or filename.
- Reindexes the time dimension to a common hourly range (ensuring all stations align in time).
- Returns the cleaned dataset.


In [2]:
def preprocess_station(file_path, date_range):
    """Open and preprocess a single NetCDF file."""
    ds = xr.open_dataset(file_path)
    
    # Clean invalid attributes
    #ds = clean_attrs(ds)

    if 'input_station_id' in ds:
        ds = ds.drop_vars('input_station_id')

    # Assign station ID from attributes or filename
    station_id = ds.attrs.get("station_id", file_path.stem)
    ds = ds.assign_coords(station_id=station_id)

    # Promote lat/lon/elevation to coordinates (if not already)
    for coord in ["latitude", "longitude", "elevation"]:
        if coord in ds and coord not in ds.coords:
            ds = ds.set_coords(coord)

    # Reindex time to common range
    target_time = pd.date_range(date_range[0], date_range[1], freq='h')
    ds = ds.reindex(time=target_time)

    return ds

    #Explain reason for reindexing and cutting out pre 1970 poor data quality
    

In [3]:
# def preprocess_station(file_path, date_range):
#     """Open and preprocess a single NetCDF file."""
#     ds = xr.open_dataset(file_path)
    
#     # Clean invalid attributes
#     #ds = clean_attrs(ds)
    
#     if 'input_station_id' in ds:
#         ds = ds.drop_vars('input_station_id')
#     # Extract station_id from attributes, coordinate, or filename
#     station_id = ds.attrs.get('station_id', None)
#     if station_id is None and 'station_id' in ds.coords:
#         station_id = str(ds.coords['station_id'].values)
#     if station_id is None:
#         station_id = file_path.stem
#     # Ensure station_id is a string (not object dtype)
#     station_id = str(station_id)
#     # Use a single 'station' coordinate and dimension
#     ds = ds.expand_dims({'station': [station_id]})
#     ds = ds.assign_coords(station=[station_id])
    
#     # Promote lat/lon/elevation to coordinates (if not already)
#     for coord in ["latitude", "longitude", "elevation"]:
#         if coord in ds and coord not in ds.coords:
#             ds = ds.set_coords(coord)
    
#     # Reindex time to common range
#     target_time = pd.date_range(date_range[0], date_range[1], freq='h')
#     ds = ds.reindex(time=target_time)
    
#     return ds
    
#     #Explain reason for reindexing and cutting out pre 1970 poor data quality

The HadISD NetCDF files store latitude, longitude, and elevation as coordinates with a singleton coordinate_length dimension. When merging multiple stations, these become coordinates of shape (station, coordinate_length). To ensure they are always available as coordinates (and not lost when selecting variables), we explicitly promote them with set_coords. After merging, we remove the unnecessary coordinate_length dimension, resulting in 1D auxiliary coordinates of shape (station,) for each station.

# Convert NetCDF to Zarr

Purpose:
- Batch-processes all NetCDF files in a directory, converting each to a Zarr store.

Key steps:
- Iterates through all .nc files in the input directory.
- Applies the preprocess_station function to each file.
- Saves each processed dataset as an individual Zarr store in the output directory.


In [4]:
def process_all_to_zarr(netcdf_dir, zarr_output_dir, date_range, scheduler='threads', num_workers=None):
    """Parallel NetCDF to Zarr conversion using dask.delayed. Returns both success boolean and status message."""
    netcdf_dir = Path(netcdf_dir)
    zarr_output_dir = Path(zarr_output_dir)
    zarr_output_dir.mkdir(parents=True, exist_ok=True)

    netcdf_files = list(netcdf_dir.glob("*.nc"))
    zarr_files = set(f.stem for f in zarr_output_dir.glob("*.zarr"))

    tasks = []
    skipped = 0

    def convert_netcdf_to_zarr(nc_file, out_path, date_range):
        try:
            ds = preprocess_station(nc_file, date_range)
            ds.to_zarr(str(out_path), mode='w')
            return (True, f"Converted: {nc_file.name} → {out_path.name}")
        except Exception as e:
            return (False, f"Failed on {nc_file.name}: {e}")

    for nc_file in netcdf_files:
        zarr_name = nc_file.stem
        out_path = zarr_output_dir / f"{zarr_name}.zarr"
        if zarr_name in zarr_files:
            # print(f"Zarr file already exists for {nc_file.name}: {out_path.name}. Skipping.")
            skipped += 1
            continue
        tasks.append(delayed(convert_netcdf_to_zarr)(nc_file, out_path, date_range))

    if not tasks:
        print(f"No new NetCDF files to convert. {skipped} already present.")
        return

    if num_workers is None:
        num_workers = multiprocessing.cpu_count() // 2

    print(f"Starting Dask parallel conversion with {num_workers} workers...")
    results = compute(*tasks, scheduler=scheduler, num_workers=num_workers)
    for success, msg in results:
        print(msg)
    converted = sum(success for success, _ in results)
    print(f"Conversion complete. {converted} new stations converted, {skipped} already present.")

In [5]:
# def process_all_to_zarr(netcdf_dir, zarr_output_dir, date_range, scheduler='threads', num_workers=None):
#     """Parallel NetCDF to Zarr conversion using dask.delayed. Returns both success boolean and status message."""
#     netcdf_dir = Path(netcdf_dir)
#     zarr_output_dir = Path(zarr_output_dir)
#     zarr_output_dir.mkdir(parents=True, exist_ok=True)

#     netcdf_files = list(netcdf_dir.glob("*.nc"))
#     zarr_files = set(f.stem for f in zarr_output_dir.glob("*.zarr"))

#     tasks = []
#     skipped = 0

#     def convert_netcdf_to_zarr(nc_file, out_path, date_range):
#         try:
#             ds = preprocess_station(nc_file, date_range)
#             # Set chunk size to half the time dimension for two chunks per variable
#             chunk_size = 236676  # 473352 // 2

#             encoding = {var: {'chunks': (chunk_size,)} for var in ds.data_vars}
#             ds.to_zarr(str(out_path), encoding=encoding, mode='w')
#             return (True, f"Converted: {nc_file.name} → {out_path.name}")
#         except Exception as e:
#             return (False, f"Failed on {nc_file.name}: {e}")

#     for nc_file in netcdf_files:
#         zarr_name = nc_file.stem
#         out_path = zarr_output_dir / f"{zarr_name}.zarr"
#         if zarr_name in zarr_files:
#             # print(f"Zarr file already exists for {nc_file.name}: {out_path.name}. Skipping.")
#             skipped += 1
#             continue
#         tasks.append(delayed(convert_netcdf_to_zarr)(nc_file, out_path, date_range))

#     if not tasks:
#         print(f"No new NetCDF files to convert. {skipped} already present.")
#         return

#     if num_workers is None:
#         num_workers = multiprocessing.cpu_count() // 2

#     print(f"Starting Dask parallel conversion with {num_workers} workers...")
#     results = compute(*tasks, scheduler=scheduler, num_workers=num_workers)
#     for success, msg in results:
#         print(msg)
#     converted = sum(success for success, _ in results)
#     print(f"Conversion complete. {converted} new stations converted, {skipped} already present.")

# Load all individual Zarr stores into a single xarray Dataset
Purpose:
- Loads all individual Zarr stores and combines them into a single xarray Dataset for analysis.

Key steps:
- Finds all .zarr stores in the specified directory.
- Uses xr.open_mfdataset to open and concatenate them along the station dimension.
- Returns the combined dataset, ready for further analysis.

In [6]:
def load_combined_dataset(zarr_dir):
    # Open all Zarr stores together
    zarr_paths = list(Path(zarr_dir).glob("*.zarr"))

    # Combine along station dimension
    ds = xr.open_mfdataset(
        zarr_paths,
        combine="nested",
        concat_dim="station",
        parallel=True,
        engine="zarr"
    )
    return ds

Combining data like this is far quicker and more resource efficient than using NetCDF files directly.

# Execute the Pre-processing and Conversion to Zarr 
We can run the `Data_Config.ipynb` to set the following:
- Date range to reindex the data
- Path to NetCDFs that need converting to Zarr
- Output location of the Zarr store

In [7]:
%run Data_Config.ipynb
print(f"NetCDF input directory: {input_dir}")
print(f"Zarr output directory: {zarr_output_dir}")
print(f"Date range: {DATE_RANGE}")

NetCDF input directory: /Users/joelmiller/HadISD_data/netcdf
Zarr output directory: /Users/joelmiller/HadISD_data/zarr
Date range: ('1970-01-01T00', '2023-12-31T23')


In [8]:
# Supress conversion to zarr warnings
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning"
warnings.filterwarnings("ignore", message=".*not part in the Zarr format 3 specification.*")
warnings.filterwarnings("ignore", message=".*vlen-utf8.*")
warnings.filterwarnings("ignore", message=".*dtype <U.*")

In [9]:
# Run parallel NetCDF-to-Zarr conversion with safe defaults
process_all_to_zarr(str(input_dir), str(zarr_output_dir), DATE_RANGE, scheduler='processes', num_workers=10)

Starting Dask parallel conversion with 10 workers...
Converted: hadisd.3.4.0.2023f_19310101-20240101_026050-99999.nc → hadisd.3.4.0.2023f_19310101-20240101_026050-99999.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_022690-99999.nc → hadisd.3.4.0.2023f_19310101-20240101_022690-99999.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_011600-99999.nc → hadisd.3.4.0.2023f_19310101-20240101_011600-99999.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_014830-99999.nc → hadisd.3.4.0.2023f_19310101-20240101_014830-99999.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_012100-99999.nc → hadisd.3.4.0.2023f_19310101-20240101_012100-99999.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_010260-99999.nc → hadisd.3.4.0.2023f_19310101-20240101_010260-99999.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_028140-99999.nc → hadisd.3.4.0.2023f_19310101-20240101_028140-99999.zarr
Converted: hadisd.3.4.0.2023f_19310101-20240101_024490-99999.nc → hadisd.3.4.0.2023f_19310101-202401

Show the combined dataset from the Zarr stores

In [10]:
ds_combined = load_combined_dataset(zarr_output_dir)
ds_combined

<xarray.Dataset> Size: 361GB
Dimensions:                (station: 851, time: 473352, flagged: 19, test: 71,
                            reporting_v: 19, reporting_t: 1116, reporting_2: 2,
                            coordinate_length: 1)
Coordinates:
    elevation              (station, coordinate_length) float64 7kB dask.array<chunksize=(1, 1), meta=np.ndarray>
    latitude               (station, coordinate_length) float64 7kB dask.array<chunksize=(1, 1), meta=np.ndarray>
    longitude              (station, coordinate_length) float64 7kB dask.array<chunksize=(1, 1), meta=np.ndarray>
    station_id             (station) <U12 41kB '024640-99999' ... '722223-13899'
  * time                   (time) datetime64[ns] 4MB 1970-01-01 ... 2023-12-3...
Dimensions without coordinates: station, flagged, test, reporting_v,
                                reporting_t, reporting_2, coordinate_length
Data variables: (12/25)
    cloud_base             (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    dewpoints              (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    flagged_obs            (station, time, flagged) float64 61GB dask.array<chunksize=(1, 29585, 3), meta=np.ndarray>
    high_cloud_cover       (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    low_cloud_cover        (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    mid_cloud_cover        (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    ...                     ...
    stnlp                  (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    temperatures           (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    total_cloud_cover      (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    wind_gust              (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    winddirs               (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
    windspeeds             (station, time) float64 3GB dask.array<chunksize=(1, 59169), meta=np.ndarray>
Attributes: (12/39)
    Conventions:                 CF-1.6
    Metadata_Conventions:        Unidata Dataset Discovery v1.0, CF Discrete ...
    acknowledgement:             RJHD was supported by the Joint BEIS/Defra M...
    cdm_data_type:               station
    creator_email:               robert.dunn@metoffice.gov.uk
    creator_name:                Robert Dunn
    ...                          ...
    station_id:                  024640-99999
    station_information:         Where station is a composite the station id ...
    summary:                     Quality-controlled, sub-daily, station datas...
    time_coverage_end:           2023-12-31T23:00Z
    time_coverage_start:         1931-01-01T12:00Z
    title:                       HadISD

In [11]:
for var in ds_combined.data_vars:
    print(f"{var}: {ds_combined[var].chunks}")

cloud_base: ((1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [ ]:
print(ds_combined.dims)

In [ ]:
print(ds_combined.station_id.values)

In [ ]:
for var in ds_combined.data_vars:
    print(f"{var} attributes: {ds_combined[var].attrs}")

In [ ]:
for var in ds_combined.data_vars:
    mv = ds_combined[var].attrs.get("missing_value")
    fv = ds_combined[var].encoding.get("_FillValue")
    if mv is not None or fv is not None:
        print(f"{var}: missing_value={mv}, _FillValue={fv}")


In [ ]:
import numpy as np

for var in ds_combined.data_vars:
    flagged_val = ds_combined[var].attrs.get('flagged_value', None)
    if flagged_val is not None:
        ds_combined[var] = ds_combined[var].where(ds_combined[var] != flagged_val, np.nan)
        print(f"Flagged values in {var} replaced with NaN: {flagged_val}")

In [ ]:
import numpy as np

for var in ds_combined.data_vars:
    # Set both _FillValue and missing_value to np.nan for consistency
    ds_combined[var].attrs['_FillValue'] = np.nan
    ds_combined[var].attrs['missing_value'] = np.nan
    # If present in encoding, update there too
    if '_FillValue' in ds_combined[var].encoding:
        ds_combined[var].encoding['_FillValue'] = np.nan
    print(f"Harmonized _FillValue and missing_value for {var} to np.nan")

In [ ]:
import numpy as np

for var in ds_combined.data_vars:
    ds_combined[var].attrs['_FillValue'] = np.nan
    ds_combined[var].attrs['missing_value'] = np.nan
    ds_combined[var].encoding['_FillValue'] = np.nan
    print(f"Set _FillValue and missing_value for {var} to np.nan")

In [ ]:
import numpy as np

def harmonise_fill_values(ds):
    for var in ds.data_vars:
        v = ds[var]

        # Standardize to NaN internally
        mv = v.attrs.get("missing_value")
        fv = v.encoding.get("_FillValue")

        # Pick one consistent value
        standard = np.nan  

        # Replace old fill values in data
        if mv is not None and not np.isnan(mv):
            v.data = v.data.where(v.data != mv, standard)
        if fv is not None and not np.isnan(fv):
            v.data = v.data.where(v.data != fv, standard)

        # Remove conflicting metadata
        v.attrs.pop("missing_value", None)
        v.encoding["_FillValue"] = standard

    return ds

ds_combined = harmonise_fill_values(ds_combined)

In [ ]:
# 2. Define your new chunking (e.g., 407 stations per chunk)
target_chunks = {'station': 407, 'time': -1}

# 3. Set up rechunker
import os
import tempfile

output_store = "/Users/joelmiller/HadISD_data/combined_407stations.zarr"
temp_store = tempfile.mkdtemp()

rechunked = rechunker.rechunk(
    ds_clean,
    target_chunks=target_chunks,    
    max_mem="12GB",  # adjust as needed
    target_store=output_store,
    temp_store=temp_store,
)

# 4. Execute the rechunking
rechunked.execute()

In [ ]:
import platform
import psutil

def print_system_specs():
    print("System Information:")
    print(f"OS: {platform.system()} {platform.release()} ({platform.version()})")
    print(f"Machine: {platform.machine()}")
    print(f"Processor: {platform.processor()}")
    print(f"CPU cores (logical): {psutil.cpu_count(logical=True)}")
    print(f"CPU cores (physical): {psutil.cpu_count(logical=False)}")
    mem = psutil.virtual_memory()
    print(f"Total RAM: {mem.total / (1024 ** 3):.2f} GB")
    print(f"Available RAM: {mem.available / (1024 ** 3):.2f} GB")
    print(f"Used RAM: {mem.used / (1024 ** 3):.2f} GB")
    print(f"RAM Usage: {mem.percent}%")

print_system_specs()

In [ ]:
import sys
print(sys.executable)

In [ ]:
import zarr
print(zarr.__version__)

In [ ]:
ds.station.values

In [ ]:
# Open the rechunked Zarr store
ds = xr.open_zarr("/users/joelmiller/HadISD_data/combined_407stations.zarr")

# Show basic info
print(ds)

# Example: list variables
print(ds.data_vars)

# Example: select data for a specific station
station_id = ds.station.values[0]  # or use a known station value
ds_station = ds.sel(station=station_id)
print(ds_station)

# Example: plot a variable
ds_station['temperature'].plot()

# Data Organization: NetCDF and Zarr stores

To keep your workflow clear and reproducible, we recommend storing both the raw NetCDF files and the processed Zarr data in separate subfolders inside your main WMO directory. For example:

- `HadISD_data/WMO_080000-099999/netcdf/` (raw NetCDF files)
- `HadISD_data/WMO_080000-099999/zarr/` (processed Zarr stores with harmonized time coordinates)

This makes it obvious which data is raw and which is ready for fast, parallel analysis.

In [ ]:
# Load and preprocess a single NetCDF file (no Zarr conversion)
netcdf_path = Path("/home/users/joel.miller/HadISD_data/netcdf/hadisd.3.4.0.2023f_19310101-20240101_501360-99999.nc")
DATE_RANGE = ("1970-01-01T00", "2023-12-31T23")
ds = preprocess_station(netcdf_path, DATE_RANGE)
ds

In [ ]:
ds = ds.expand_dims({'station': [station_id]})
ds

In [ ]:
# Extract station_id from the dataset attributes or coordinate
station_id = ds.attrs.get('station_id', None)
if station_id is None and 'station_id' in ds.coords:
    station_id = str(ds.coords['station_id'].values)
print(f"Expanding dims with station_id: {station_id}")
ds = ds.expand_dims({'station': [station_id]})
ds

In [ ]:
ds.station.values

In [ ]:
 # Convert station coordinate to integer if possible, else to fixed-length string
import numpy as np
station_vals = ds_combined.station.values
try:
    # Try converting to integer
    station_int = station_vals.astype(int)
    ds_combined = ds_combined.assign_coords(station=station_int)
    print("Converted station coordinate to integer")
except Exception:
    # Fallback: convert to fixed-length Unicode string
    station_str = station_vals.astype('U12')  # adjust length as needed
    ds_combined = ds_combined.assign_coords(station=station_str)
    print("Converted station coordinate to fixed-length string.")
# Check new dtype
print(ds_combined.station.dtype)
print(ds_combined.station.values)

In [ ]:
ds_combined.station.values

In [ ]:
import xarray as xr
import pandas as pd

date_range = ("1970-01-01T00", "2023-12-31T23")
netcdf_dir = input_dir
netcdf_files = [str(f) for f in netcdf_dir.glob("*.nc")]

def preprocess(ds):
    # Drop problematic variables
    if 'input_station_id' in ds:
        ds = ds.drop_vars('input_station_id')
    # Assign station_id
    station_id = ds.attrs.get("station_id", None)
    if station_id is None and 'station_id' in ds.coords:
        station_id = str(ds.coords['station_id'].values)
    if station_id is None:
        station_id = ds.encoding.get("source", "unknown")
    ds = ds.assign_coords(station_id=station_id)
    # Promote lat/lon/elevation
    for coord in ["latitude", "longitude", "elevation"]:
        if coord in ds and coord not in ds.coords:
            ds = ds.set_coords(coord)
    # Reindex time
    target_time = pd.date_range(date_range[0], date_range[1], freq='h')
    ds = ds.reindex(time=target_time)
    return ds

# Use open_mfdataset with preprocess
combined = xr.open_mfdataset(
    netcdf_files,
    preprocess=preprocess,
    combine="nested",
    concat_dim="station_id",
    parallel=True,
    engine="netcdf4"
)

combined.to_netcdf("processed_combined.nc")